# Phase 02: Feature Engineering & Target Analysis 

* **Technical Literature Review:** Research and select specific preprocessing techniques (e.g., handling extreme skewness, high cardinality, and missingness) identified during the Phase 1 exploration.
* **Feature Prioritization:** Compile a list of potential new variables and rank them by their expected predictive power and implementation complexity.
* **Feature Engineering:** Derive new features based on the initial hypotheses (such as temporal extractions from `fecha` or behavioral ratios) to enhance the model's predictive performance.
* **Target Characterization:** Analyze the distribution of the `fraude` variable and its specific relationship with other variables.
* **Variable Relevance:** Use statistical tests (e.g., Chi-Squared for categorical data, ANOVA for numerical data) to quantify the relationship between features and the target.
* **Hypothesis Formulation:** Create a list of "Risk Indicators" (e.g., "Transactions above amount $X$ are $Y\%$ more likely to be fraud").
* **Feature Selection:** If necessary, other feature selection could be done after this previous steps. 

# 1. Technical Literature Review and Proposed Preprocessing Strategy

Based on a review of established literature in statistical learning, credit risk modeling, and production ML systems — including:

* **The Elements of Statistical Learning: Data Mining, Inference, and Prediction** – Trevor Hastie, Robert Tibshirani, Jerome Friedman (2001) – on transformations, regularization, and model behavior under skewed distributions.
* **Pattern Recognition and Machine Learning** – Christopher M. Bishop (2006) – on probabilistic modeling, likelihood-based methods, and feature transformations.
* **Credit Risk Scorecards: Developing and Implementing Intelligent Credit Scoring** – Naeem Siddiqi (2006) – on Weight of Evidence (WOE), Information Value (IV), binning strategies, and handling missingness in risk modeling.
* **Designing Machine Learning Systems: An Iterative Process for Production-Ready Applications** – Chip Huyen (2022) – on practical data preparation, feature engineering, model selection, and production considerations.


we identified a set of preprocessing techniques appropriate for the structural issues observed during Phase 1 (EDA), particularly skewness, high cardinality, missingness, imbalance, and correlation.

The following table summarizes the proposed solutions aligned with best practices from the literature:


| Problem Identified                             | Variables Affected         | Technical Risk                                          | Proposed Solutions                                                                                                        | Business Justification                           |
| ---------------------------------------------- | -------------------------- | ------------------------------------------------------- | ------------------------------------------------------------------------------------------------------------------------- | ------------------------------------------------ |
| **Target imbalance (5% fraud)**                | fraude                     | Model biased toward majority class; accuracy misleading | Cost-sensitive learning; class weights; threshold optimization based on profit; PR-AUC; Expected Profit curve             | Fraud cost is asymmetric (−100% vs +25%)         |
| **High missingness (72.6%)**                   | o                          | Imputation may hide signal                              | Missing indicator; missing as separate bin (WOE); compare imputation strategies; tree models with native missing handling | Missingness likely behavioral                    |
| **Moderate missingness (8.7%)**                | b, c                       | Bias or instability                                     | Missing indicators; median imputation; WOE bin with missing category                                                      | Incomplete information may correlate with fraud  |
| **Unique identifier (100% unique)**            | k                          | Leakage / overfitting                                   | Drop variable                                                                                                             | No predictive generalization                     |
| **High zero concentration**                    | e, f, m, h                 | Structural zero vs true zero unclear                    | Zero indicators; separate bin for zero; two-part modeling; log1p transformation                                           | Zero may represent absence of activity           |
| **Extreme skewness (very high γ1)**            | e, f                       | Linear instability; tail dominance                      | Log transform; Box–Cox; Yeo–Johnson; quantile transformation; WOE binning; percentile bins                                | Stabilizes model; captures non-linear fraud risk |
| **Heavy tail in monetary variable**            | monto                      | Extreme values dominate loss                            | Log transform; Box–Cox / Yeo–Johnson; percentile binning; top-value flag; interaction terms                               | High-value fraud drives economic loss            |
| **Correlation clusters**                       | (d, m), (e, monto), (f, l) | Multicollinearity in LR                                 | VIF analysis; drop redundancy; optional PCA; tree models handle naturally                                                 | Avoid unstable coefficients                      |
| **Categorical imbalance (85.7% one category)** | a                          | Rare categories unstable                                | Rare grouping; WOE encoding; frequency encoding                                                                           | Minority categories may carry higher fraud rate  |
| **High cardinality (~8,250 distinct values)**  | j                          | One-hot infeasible; sparse representation; overfitting  | Rare grouping; frequency encoding; target encoding (smoothed); hashing trick                                              | High-cardinality categories may concentrate fraud in specific values |
| **Temporal variable unused**                   | fecha                      | Loss of behavioral signal                               | Extract hour, weekday; cyclic encoding                                                                                    | Fraud often temporal                             |
| **Behavior–amount interaction ignored**        | e, monto                   | Interaction effect missed                               | Ratio feature; interaction terms                                                                                          | Abnormal intensity patterns signal fraud         |
| **Outliers removal risk**                      | e, f, monto                | Removing fraud signal                                   | Avoid trimming; prefer transformation; robust scaling                                                                     | Fraud often lives in tails                       |
| **Dataset clean (no duplicates)**              | Dataset                    | No issue                                                | No action required                                                                                                        | Data integrity confirmed                         |


So, based on this, follow a table of original and new variables to be analysed:

| Feature | Type               | Action | Justification                                                                    |
| ------- | ------------------ | ------ | -------------------------------------------------------------------------------- |
| a       | Cat                | keep   | Categorical variable; potential fraud signal after encoding                      |
| b       | Numeric            | keep   | Missingness handled via engineered indicator                                     |
| c       | Numeric            | keep   | Missingness handled via engineered indicator                                     |
| d       | Numeric            | keep   | Potential predictive signal; correlation manageable                              |
| e       | Numeric            | keep   | Strong behavioral variable; heavy skew handled via transformation                |
| f       | Numeric            | keep   | Behavioral variable; heavy skew handled via transformation                       |
| g       | Cat                | keep   | Geographic variable; possible fraud concentration                                |
| h       | Numeric            | keep   | Zero concentration may carry signal                                              |
| j       | Cat                | keep   | High cardinality; requires encoding (freq / target enc) |
| k       | Numeric            | drop   | Identifier; no predictive value; leakage risk                                    |
| l       | Numeric            | keep   | Correlated but potentially informative                                           |
| m       | Numeric            | keep   | Behavioral signal candidate                                                      |
| n       | Cat                | keep   | Binary variable; model-ready                                                     |
| o       | Numeric            | keep   | Missingness likely predictive                                                    |
| p       | Cat                | keep   | Binary categorical variable                                                      |
| fecha   | Numeric (datetime) | keep   | Used for temporal feature extraction                                             |
| monto   | Numeric            | keep   | Core economic driver of profit/loss                                              |
| score   | Numeric            | keep   | Pre-existing risk score; verify no leakage                                       |
| fraude  | Numeric            | keep   | Target variable                                                                  |



# Newly Engineered Variables

The list of possible preprocessing and feature engineering techniques identified during the literature review is extensive. Continuous variables can be transformed in multiple ways (logarithmic, Box–Cox, Yeo–Johnson, binning, percentile flags, interaction terms, etc.), and categorical variables can be encoded using different strategies depending on the modeling family.

Not all transformations are required simultaneously. Some techniques are model-dependent (e.g., binning and WOE are more relevant for linear models, while tree-based models naturally capture non-linearities). Additionally, certain interaction features may emerge during experimentation rather than being predefined. Specially, for continuos, binned varaible was crated for WOE that have special use for fraud detection according review.

The goal here is to create features and then choose the best ones according to their correlation with the target or according to their strength within the model.

For this reason, the table below consolidates the potential engineered variables to be created, the justification for each transformation, the modeling family for which it is most relevant (ML affinity):


| Feature / Transformation | Derived From | Type         | Justification                                                              | ML Affinity |
| ------------------------ | ------------ | ------------ | -------------------------------------------------------------------------- | ----------- |
| b_is_missing             | b            | Binary (Cat) | Missingness may represent abnormal behavior                                | Both        |
| c_is_missing             | c            | Binary (Cat) | Missingness may carry structural signal                                    | Both        |
| o_is_missing             | o            | Binary (Cat) | 72.6% missing strongly suggests segmentation                               | Both        |
| e_is_zero                | e            | Binary (Cat) | Structural inactivity indicator                                            | Both        |
| f_is_zero                | f            | Binary (Cat) | Behavioral anomaly detection                                               | Both        |
| m_is_zero                | m            | Binary (Cat) | Behavioral inactivity segmentation                                         | Both        |
| h_is_zero                | h            | Binary (Cat) | Inactivity pattern detection                                               | Both        |
| d_top1_flag              | d            | Binary (Cat) | Tail anomaly detection                                                     | Both        |
| e_top1_flag              | e            | Binary (Cat) | Tail anomaly detection                                                     | Both        |
| f_top1_flag              | f            | Binary (Cat) | Tail anomaly detection                                                     | Both        |
| monto_top1_flag          | monto        | Binary (Cat) | High financial exposure                                                    | Both        |
| bin_c                    | c            | Cat          | Required for IV                                                            | Linear      |
| bin_d                    | d            | Cat          | Required for IV                                                            | Linear      |
| bin_e                    | e            | Cat          | Required for IV                                                            | Linear      |
| bin_f                    | f            | Cat          | Required for IV                                                            | Linear      |
| bin_k                    | f            | Cat          | Required for IV                                                            | Linear      |
| bin_l                    | f            | Cat          | Required for IV                                                            | Linear      |
| bin_m                    | f            | Cat          | Required for IV                                                            | Linear      |
| bin_monto                | monto        | Cat          | Economic segmentation for IV                                               | Linear      |
| bin_score                | monto        | Cat          | Economic segmentation for IV                                               | Linear      |
| rare_group_a             | a            | Cat          | Stabilize sparse categories                                                | Both        |
| j_rare_grouped           | j            | Cat          | Collapse rare categories (< 1% freq) before encoding; reduces cardinality  | Both        |
| j_freq_encoded           | j            | Numeric      | Frequency encoding; replaces category with occurrence rate; leakage-free   | Both        |
| weekday                  | fecha        | Cat          | Weekly fraud pattern                                                       | Both        |
| is_weekend               | fecha        | Binary (Cat) | Weekend fraud spike                                                        | Both        |
| hour                     | fecha        | Numeric      | Time-of-day pattern                                                        | Both        |
| hour_sin                 | fecha        | Numeric      | Cyclic encoding for LR                                                     | Linear      |
| hour_cos                 | fecha        | Numeric      | Cyclic encoding complement                                                 | Linear      |
| monto_per_e              | monto / e    | Numeric      | Abnormal intensity detection                                               | Both        |
| interaction_monto_e      | monto × e    | Numeric      | Non-linear interaction capture                                             | Linear      |
| log_e                    | e            | Numeric      | Skew correction                                                            | Linear      |
| log_f                    | f            | Numeric      | Skew correction                                                            | Linear      |
| log_monto                | monto        | Numeric      | Heavy-tail normalization                                                   | Linear      |
| boxcox_e                 | e            | Numeric      | Parametric skew normalization                                              | Linear      |
| boxcox_f                 | f            | Numeric      | Alternative skew correction                                                | Linear      |
| boxcox_monto             | monto        | Numeric      | Monetary stabilization                                                     | Linear      |
| yeojohnson_e             | e            | Numeric      | Robust skew correction                                                     | Linear      |
| yeojohnson_f             | f            | Numeric      | Robust skew correction                                                     | Linear      |
| PCA_behavior_cluster     | d, m         | Numeric      | Reduce multicollinearity in behavioral cluster                             | Linear      |
| PCA_amount_cluster       | e, monto     | Numeric      | Extract latent monetary-behavior factor                                    | Linear      |


# Setup and Imports

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from src.etl import load_raw_mercadolibre_dataset, save_processed_features
from preprocessing.feature_engineering import FeatureEngineer
from preprocessing.visualization import plot_feature, save_all_feature_plots
from preprocessing.visualization import save_all_feature_plots
from preprocessing.statistics import FeatureTargetStatistics
from preprocessing.visualization import plot_feature, save_all_feature_plots, plot_numeric_feature, plot_categorical_feature
from preprocessing.utils import build_feature_info

# directories
from src import RESULTS_DIR
IMAGES_DIR = RESULTS_DIR / "images"
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

IMAGES_CATEGORY_DIR = IMAGES_DIR / "category"
IMAGES_CATEGORY_DIR.mkdir(parents=True, exist_ok=True)

IMAGES_NUMERIC_DIR = IMAGES_DIR / "numeric"
IMAGES_NUMERIC_DIR.mkdir(parents=True, exist_ok=True)


# Initial Analysis

In [2]:
df_raw = load_raw_mercadolibre_dataset()
print("Data loaded:")
display(df_raw.head())

Loading raw MercadoLibre dataset from dataset\raw\MercadoLibre Data Scientist Technical Challenge - Dataset.csv...
Raw MercadoLibre dataset loaded successfully from dataset\raw\MercadoLibre Data Scientist Technical Challenge - Dataset.csv as a pandas DataFrame.
Data loaded:


,a,b,c,d,e,f,g,h,j,k,l,m,n,o,p,fecha,monto,score,fraude
0,4,0.6812,50084.12,50.0,0.000000,20.0,AR,1,cat_d26ab52,0.365475,2479.0,952.0,1,NaN,Y,2020-03-20 09:28:19,57.63,100,0
1,4,0.6694,66005.49,0.0,0.000000,2.0,AR,1,cat_ea962fb,0.612728,2603.0,105.0,1,Y,Y,2020-03-09 13:58:28,40.19,25,0
2,4,0.4718,7059.05,4.0,0.463488,92.0,BR,25,cat_4c2544e,0.651835,2153.0,249.0,1,Y,Y,2020-04-08 12:25:55,5.77,23,0
3,4,0.7260,10043.10,24.0,0.046845,43.0,BR,43,cat_1b59ee3,0.692728,4845.0,141.0,1,N,Y,2020-03-14 11:46:13,40.89,23,0
4,4,0.7758,16584.42,2.0,0.154616,54.0,BR,0,cat_9bacaa5,0.201354,2856.0,18.0,1,Y,N,2020-03-23 14:17:13,18.98,71,0


# Feature Engineering


## Calculating features 

The fucntion below will calculate all the considered features, beyond categorize this by categorical or numerical. 

In [3]:
original_features_config = {
    "a": {
        "dtype": "categorical",
        "action": "keep"
    },
    "b": {
        "dtype": "continuous",
        "action": "keep"
    },
    "c": {
        "dtype": "continuous",
        "action": "keep"
    },
    "d": {
        "dtype": "continuous",
        "action": "keep"
    },
    "e": {
        "dtype": "continuous",
        "action": "keep"
    },
    "f": {
        "dtype": "continuous",
        "action": "keep"
    },
    "g": {
        "dtype": "categorical",
        "action": "keep"
    },
    "h": {
        "dtype": "continuous",
        "action": "keep"
    },
    "j": {
        "dtype": "categorical",
        "action": "keep"
    },
    "k": {
        "dtype": "continuous",
        "action": "drop"
    },
    "l": {
        "dtype": "continuous",
        "action": "keep"
    },
    "m": {
        "dtype": "continuous",
        "action": "keep"
    },
    "n": {
        "dtype": "categorical",
        "action": "keep"
    },
    "o": {
        "dtype": "categorical",
        "action": "keep"
    },
    "p": {
        "dtype": "categorical",
        "action": "keep"
    },
    "fecha": {
        "dtype": "datetime",
        "action": "keep"
    },
    "monto": {
        "dtype": "continuous",
        "action": "keep"
    },
    "score": {
        "dtype": "continuous",
        "action": "keep"
    },
    "fraude": {
        "dtype": "continuous",
        "action": "keep"
    },  # target
}


feature_engineering_config = {
    # Missing flags (categorical/binary)
    "b_is_missing": {
        "dtype": "categorical",
        "config": {
            "process": "missing_flag",
            "col": "b",
            "new_col": "b_is_missing"
        }
    },
    "c_is_missing": {
        "dtype": "categorical",
        "config": {
            "process": "missing_flag",
            "col": "c",
            "new_col": "c_is_missing"
        }
    },
    "o_is_missing": {
        "dtype": "categorical",
        "config": {
            "process": "missing_flag",
            "col": "o",
            "new_col": "o_is_missing"
        }
    },

    # Zero flags (categorical/binary)
    "e_is_zero": {
        "dtype": "categorical",
        "config": {
            "process": "zero_flag",
            "col": "e",
            "new_col": "e_is_zero"
        }
    },
    "f_is_zero": {
        "dtype": "categorical",
        "config": {
            "process": "zero_flag",
            "col": "f",
            "new_col": "f_is_zero"
        }
    },
    "m_is_zero": {
        "dtype": "categorical",
        "config": {
            "process": "zero_flag",
            "col": "m",
            "new_col": "m_is_zero"
        }
    },
    "h_is_zero": {
        "dtype": "categorical",
        "config": {
            "process": "zero_flag",
            "col": "h",
            "new_col": "h_is_zero"
        }
    },

    # Tail flags (categorical/binary)
    "c_top1_flag": {
        "dtype": "categorical",
        "config": {
            "process": "top_percentile_flag",
            "col": "c",
            "new_col": "c_top1_flag",
            "params": {
                "percentile": 0.99
            }
        },
    },
    "e_top1_flag": {
        "dtype": "categorical",
        "config": {
            "process": "top_percentile_flag",
            "col": "e",
            "new_col": "e_top1_flag",
            "params": {
                "percentile": 0.99
            }
        },
    },
    "f_top1_flag": {
        "dtype": "categorical",
        "config": {
            "process": "top_percentile_flag",
            "col": "f",
            "new_col": "f_top1_flag",
            "params": {
                "percentile": 0.99
            }
        },
    },
    "monto_top1_flag": {
        "dtype": "categorical",
        "config": {
            "process": "top_percentile_flag",
            "col": "monto",
            "new_col": "monto_top1_flag",
            "params": {
                "percentile": 0.99
            }
        },
    },

    # Binning (categorical)
    "bin_c": {
        "dtype": "categorical",
        "config": {
            "process": "quantile_bins_dropna",
            "col": "c",
            "new_col": "bin_c",
            "params": {
                "n_bins": 10,
                "add_missing_bin": True
            }
        },
    },
    "bin_d": {
        "dtype": "categorical",
        "config": {
            "process": "quantile_bins_dropna",
            "col": "d",
            "new_col": "bin_d",
            "params": {
                "n_bins": 10,
                "add_missing_bin": True
            }
        },
    },
    "bin_e": {
        "dtype": "categorical",
        "config": {
            "process": "quantile_bins_dropna",
            "col": "e",
            "new_col": "bin_e",
            "params": {
                "n_bins": 10,
                "add_missing_bin": True
            }
        },
    },
    "bin_f": {
        "dtype": "categorical",
        "config": {
            "process": "quantile_bins_dropna",
            "col": "f",
            "new_col": "bin_f",
            "params": {
                "n_bins": 10,
                "add_missing_bin": True
            }
        },
    },
    "bin_h": {
        "dtype": "categorical",
        "config": {
            "process": "quantile_bins_dropna",
            "col": "h",
            "new_col": "bin_h",
            "params": {
                "n_bins": 10,
                "add_missing_bin": True
            }
        },
    },
    "bin_k": {
        "dtype": "categorical",
        "config": {
            "process": "quantile_bins_dropna",
            "col": "k",
            "new_col": "bin_k",
            "params": {
                "n_bins": 10,
                "add_missing_bin": True
            }
        },
    },
    "bin_l": {
        "dtype": "categorical",
        "config": {
            "process": "quantile_bins_dropna",
            "col": "l",
            "new_col": "bin_l",
            "params": {
                "n_bins": 10,
                "add_missing_bin": True
            }
        },
    },
    "bin_m": {
        "dtype": "categorical",
        "config": {
            "process": "quantile_bins_dropna",
            "col": "m",
            "new_col": "bin_m",
            "params": {
                "n_bins": 10,
                "add_missing_bin": True
            }
        },
    },
    "bin_score": {
        "dtype": "categorical",
        "config": {
            "process": "quantile_bins_dropna",
            "col": "score",
            "new_col": "bin_score",
            "params": {
                "n_bins": 10,
                "add_missing_bin": True
            }
        },
    },
    "bin_monto": {
        "dtype": "categorical",
        "config": {
            "process": "quantile_bins_dropna",
            "col": "monto",
            "new_col": "bin_monto",
            "params": {
                "n_bins": 10,
                "add_missing_bin": True
            }
        },
    },

    # Rare grouping (categorical)
    "a_rare_grouped": {
        "dtype": "categorical",
        "config": {
            "process": "rare_grouping",
            "col": "a",
            "new_col": "a_rare_grouped",
            "params": {
                "min_freq": 0.01,
                "other_label": "__OTHER__"
            }
        },
    },

    # High-cardinality encoding for j
    "j_rare_grouped": {
        "dtype": "categorical",
        "config": {
            "process": "rare_grouping",
            "col": "j",
            "new_col": "j_rare_grouped",
            "params": {
                "min_freq": 0.01,
                "other_label": "__OTHER__"
            }
        },
    },
    "j_freq_encoded": {
        "dtype": "continuous",
        "config": {
            "process": "frequency_encoding",
            "col": "j",
            "new_col": "j_freq_encoded"
        }
    },

    # Datetime-derived discrete/categorical
    "hour": {
        "dtype": "categorical",
        "config": {
            "process": "hour",
            "col": "fecha",
            "new_col": "hour"
        }
    },
    "weekday": {
        "dtype": "categorical",
        "config": {
            "process": "weekday",
            "col": "fecha",
            "new_col": "weekday"
        }
    },
    "is_weekend": {
        "dtype": "categorical",
        "config": {
            "process": "is_weekend",
            "col": "fecha",
            "new_col": "is_weekend"
        }
    },

    # Datetime-derived continuous (cyclic)
    "hour_sin": {
        "dtype": "continuous",
        "config": {
            "process": "hour_cyclic",
            "col": "fecha",
            "new_col": "hour_sin",
            "params": {
                "component": "sin"
            }
        }
    },
    "hour_cos": {
        "dtype": "continuous",
        "config": {
            "process": "hour_cyclic",
            "col": "fecha",
            "new_col": "hour_cos",
            "params": {
                "component": "cos"
            }
        }
    },

    # Ratio (continuous)
    "monto_per_e": {
        "dtype": "continuous",
        "config": {
            "process": "ratio",
            "col": ("monto",
            "e"),
            "new_col": "monto_per_e"
        }
    },

    # Log transforms (continuous)
    "log_e": {
        "dtype": "continuous",
        "config": {
            "process": "log_feature",
            "col": "e",
            "new_col": "log_e"
        }
    },
    "log_f": {
        "dtype": "continuous",
        "config": {
            "process": "log_feature",
            "col": "f",
            "new_col": "log_f"
        }
    },
    "log_monto": {
        "dtype": "continuous",
        "config": {
            "process": "log_feature",
            "col": "monto",
            "new_col": "log_monto"
        }
    },

    # Power transforms (continuous)
    "yeojohnson_e": {
        "dtype": "continuous",
        "config": {
            "process": "yeojohnson",
            "col": "e",
            "new_col": "yeojohnson_e"
        }
    },
    "yeojohnson_f": {
        "dtype": "continuous",
        "config": {
            "process": "yeojohnson",
            "col": "f",
            "new_col": "yeojohnson_f"
        }
    },
    "boxcox_e": {
        "dtype": "continuous",
        "config": {
            "process": "boxcox",
            "col": "e",
            "new_col": "boxcox_e"
        }
    },
    "boxcox_f": {
        "dtype": "continuous",
        "config": {
            "process": "boxcox",
            "col": "f",
            "new_col": "boxcox_f"
        }
    },
    "boxcox_monto": {
        "dtype": "continuous",
        "config": {
            "process": "boxcox",
            "col": "monto",
            "new_col": "boxcox_monto"
        }
    },

    # Interaction (continuous)
    "interaction_monto_e": {
        "dtype": "continuous",
        "config": {
            "process": "interaction_mul",
            "col": ("monto",
            "e"),
            "new_col": "interaction_monto_e"
        }
    },

    # PCA components (continuous) - note: creates pca_beh_1, pca_beh_2, etc.
    "pca_beh": {
        "dtype": "continuous",
        "config": {
            "process": "pca_components",
            "col": ("d",
            "m"),
            "new_col": "pca_beh",
            "params": {
                "n_components": 2
            }
        }
    },
    "pca_amt": {
        "dtype": "continuous",
        "config": {
            "process": "pca_components",
            "col": ("e",
            "monto"),
            "new_col": "pca_amt",
            "params": {
                "n_components": 2
            }
        }
    },
}


In [4]:
from preprocessing.feature_engineering import FeatureEngineer
import pandas as pd

fe = FeatureEngineer(df_raw)

# Convert dict -> list (what FeatureEngineer expects)
feature_config = [v["config"] for v in feature_engineering_config.values()]

df_features = fe.apply_transformations(feature_config)

print(f"Feature Processed: Shape: {df_features.shape}")
display(df_features.head())

Feature Processed: Shape: (150000, 43)


,b_is_missing,c_is_missing,o_is_missing,e_is_zero,f_is_zero,m_is_zero,h_is_zero,c_top1_flag,e_top1_flag,f_top1_flag,...,yeojohnson_e,yeojohnson_f,boxcox_e,boxcox_f,boxcox_monto,interaction_monto_e,pca_beh_1,pca_beh_2,pca_amt_1,pca_amt_2
0,0,0,1,1,0,0,0,0,0,0,...,-0.000000,2.877011,-9.852857,2.564029,3.538244,0.000000,652.891931,4.302738,14.107048,-0.208704
1,0,0,0,1,0,0,0,0,0,0,...,-0.000000,1.076270,-9.852857,1.692154,3.262062,0.000000,-195.374779,-14.474591,-3.332946,-0.223462
2,0,0,0,0,0,0,0,0,0,0,...,0.207747,4.168008,-0.753790,3.329535,1.651222,2.674325,-51.325147,-15.779762,-37.753326,0.210900
3,0,0,0,0,0,0,0,0,0,0,...,0.042222,3.527730,-2.829562,2.951401,3.275450,1.915474,-158.515451,8.183518,-2.632986,-0.176025
4,0,0,0,0,0,0,1,0,0,0,...,0.112347,3.720519,-1.778992,3.066390,2.664765,2.934616,-282.242132,-9.272380,-24.543069,-0.086793


Create a table with the complete data (original, and features), and a dataframe with the dtype descriptions. 

In [7]:
df_complete_data, df_info_variables, continuous_cols, categorical_cols = build_feature_info(
    original_features_config=original_features_config,
    feature_engineering_config=feature_engineering_config,
    df_raw=df_raw,
    df_features=df_features,
)

print(f"Number of original features: {len(df_raw.columns)}")
print(f"Number of engineered features: {len([c for c in df_features.columns if c not in df_raw.columns])}")
print(f"Total number of features: {len(df_complete_data.columns)}")
display(df_info_variables)

save_processed_features(df_complete_data)


Number of original features: 19
Number of engineered features: 43
Total number of features: 62


,feature,type,source
0,a,categorical,original
1,b,continuous,original
2,c,continuous,original
3,d,continuous,original
4,e,continuous,original
...,...,...,...
56,interaction_monto_e,continuous,engineered
57,pca_beh_1,continuous,engineered
58,pca_beh_2,continuous,engineered
59,pca_amt_1,continuous,engineered


Processed features saved to dataset\processed\processed_features.csv


# Features / Target Associations Tests


After generating all these features, we will create and analyze the relationship between each variable (or combinations of them) and the main target.

For this purpose, we created a set of **statistical tests** to evaluate the relationship between the target and each variable.

For **continuous variables**, we used the following metrics: **KS Statistic (Kolmogorov–Smirnov)**, **Mann–Whitney U Test**, **Point-Biserial Correlation**, **Cohen’s d**, and **ROC-AUC**.

For **categorical variables**, we used the following metrics: **Chi-Squared Test**, **Cramér's V**, and **Information Value (IV) with Weight of Evidence (WoE)**.

Additionally, to support the analysis, we plotted graphs showing the behavior of each variable against the fraud target.

All images for each variable (both numerical and categorical) were saved in the folder **results/images**. Each image also includes the values of the calculated metrics.


In [8]:
# Initialize statistics engine
fts = FeatureTargetStatistics(
    df=df_complete_data,
    target_col="fraude",
    alpha=0.05
)


Calculate continuos statistics

In [9]:
df_cont_stats = fts.compute_many(
    feature_cols=continuous_cols,
    feature_type="continuous",
    return_type="series"
)

display(df_cont_stats)

filedir = RESULTS_DIR / "stats_fraud_continuos.csv"
df_cont_stats.to_csv(RESULTS_DIR / "stats_fraud_continuos.csv")
print(f"Results for continues saved in {filedir}")


,feature_type,target,alpha,mw_p,ks,ks_p,r_pb,r_pb_p,auc,auc_abs,cohens_d,logit_beta,logit_or,logit_p,interpretation
feature,,,,,,,,,,,,,,,
b,continuous,fraude,0.05,1.189799e-66,0.103831,5.067966e-60,0.038919,4.389088e-47,0.562496,0.562496,0.180973,1.561759e+00,4.767198e+00,2.529703e-47,mann_w_u: 0.000 (Significant) | ks: KS=0.104 (...
c,continuous,fraude,0.05,1.996269e-03,0.050570,1.523607e-14,0.033229,8.706510e-35,0.511200,0.511200,0.154483,1.285796e-07,1.000000e+00,2.380682e-33,mann_w_u: 0.002 (Significant) | ks: KS=0.051 (...
d,continuous,fraude,0.05,5.197785e-254,0.166351,3.842561e-172,-0.077807,1.316663e-199,0.384670,0.615330,-0.358329,-1.990222e-02,9.802945e-01,5.802917e-191,mann_w_u: 0.000 (Significant) | ks: KS=0.166 (...
e,continuous,fraude,0.05,1.751426e-27,0.084407,1.306773e-44,-0.002261,3.811651e-01,0.464400,0.535600,-0.010375,-5.006218e-02,9.511703e-01,6.249052e-02,mann_w_u: 0.000 (Significant) | ks: KS=0.084 (...
f,continuous,fraude,0.05,0.000000e+00,0.277833,0.000000e+00,-0.010489,4.858621e-05,0.316882,0.683118,-0.048128,-6.035608e-03,9.939826e-01,7.193540e-75,mann_w_u: 0.000 (Significant) | ks: KS=0.278 (...
h,continuous,fraude,0.05,1.238310e-65,0.085698,5.651399e-46,-0.034864,1.427448e-41,0.441542,0.558458,-0.160063,-1.245789e-02,9.876194e-01,3.000214e-41,mann_w_u: 0.000 (Significant) | ks: KS=0.086 (...
l,continuous,fraude,0.05,0.000000e+00,0.253039,0.000000e+00,-0.117023,0.000000e+00,0.324672,0.675328,-0.540631,-4.041204e-04,9.995960e-01,0.000000e+00,mann_w_u: 0.000 (Significant) | ks: KS=0.253 (...
m,continuous,fraude,0.05,0.000000e+00,0.230576,0.000000e+00,-0.093437,2.645486e-287,0.341471,0.658529,-0.430894,-1.943762e-03,9.980581e-01,3.550441e-271,mann_w_u: 0.000 (Significant) | ks: KS=0.231 (...
monto,continuous,fraude,0.05,5.049078e-84,0.098947,3.554151e-61,0.073784,4.363819e-180,0.566421,0.566421,0.339466,1.877463e-03,1.001879e+00,2.069168e-118,mann_w_u: 0.000 (Significant) | ks: KS=0.099 (...


Results for continues saved in notebooks\results\stats_fraud_continuos.csv


Calculate categorical statistics

In [10]:
df_cat_stats = fts.compute_many(
    feature_cols=categorical_cols,
    feature_type="categorical",
    return_type="series",
    iv_max_bins=20,        # optional: groups rare levels into '__OTHER__'
)

display(df_cat_stats)

filedir = RESULTS_DIR / "stats_fraud_categorical.csv"
df_cat_stats.to_csv(filedir, index=True)
print(f"Results for categorical saved in {filedir}")



,feature_type,target,alpha,chi2_p,cramers_v,iv,logit_lr_p,interpretation
feature,,,,,,,,
a,categorical,fraude,0.05,2.371873e-126,0.062420,0.066327,1.152483e-108,chi2: 0.000 (Significant) | Cramers_v: V=0.062...
g,categorical,fraude,0.05,7.757972e-63,0.053889,0.068969,7.582888e-73,chi2: 0.000 (Significant) | Cramers_v: V=0.054...
j,categorical,fraude,0.05,2.634339e-112,0.277732,0.082284,3.636596e-123,chi2: 0.000 (Significant) | Cramers_v: V=0.278...
n,categorical,fraude,0.05,0.000000e+00,0.167728,0.361450,0.000000e+00,chi2: 0.000 (Significant) | Cramers_v: V=0.168...
o,categorical,fraude,0.05,0.000000e+00,0.225673,0.464029,0.000000e+00,chi2: 0.000 (Significant) | Cramers_v: V=0.226...
p,categorical,fraude,0.05,0.000000e+00,0.106800,0.245520,0.000000e+00,chi2: 0.000 (Significant) | Cramers_v: V=0.107...
b_is_missing,categorical,fraude,0.05,3.583574e-14,0.019559,0.007246,2.735730e-13,chi2: 0.000 (Significant) | Cramers_v: V=0.020...
c_is_missing,categorical,fraude,0.05,3.583574e-14,0.019559,0.007246,2.735730e-13,chi2: 0.000 (Significant) | Cramers_v: V=0.020...
o_is_missing,categorical,fraude,0.05,0.000000e+00,0.219796,0.874986,0.000000e+00,chi2: 0.000 (Significant) | Cramers_v: V=0.220...


Results for categorical saved in notebooks\results\stats_fraud_categorical.csv


In [11]:
# 2) WoE bins (one table per feature) -> single CSV (stacked)
woe_tables = []

for col in categorical_cols:
    res = fts.compute(
        feature_col=col,
        feature_type="categorical",
        return_type="dict",
        include_woe_table=True,  # <-- returns the WoE table DataFrame
        iv_max_bins=20,
    )

    woe_df = res["metrics"]["information_value"].get("woe_table")
    if woe_df is None or len(woe_df) == 0:
        continue

    woe_df = woe_df.copy()
    woe_df.insert(0, "feature", col)  # tag feature name
    woe_tables.append(woe_df)

if woe_tables:
    df_woe_all = pd.concat(woe_tables, ignore_index=True)

    woe_filedir = RESULTS_DIR / "woe_bins_categorical.csv"
    df_woe_all.to_csv(woe_filedir, index=False)
    print(f"WoE bins saved in {woe_filedir}")

    print(f"Sample of df_woe_all. Total size: {df_woe_all.shape}")
    display(df_woe_all.head(20))
else:
    print("No WoE tables were generated (check missing data / single-class / too few rows).")

WoE bins saved in notebooks\results\woe_bins_categorical.csv
Sample of df_woe_all. Total size: (207, 9)


,feature,x,count,bad,good,dist_good,dist_bad,woe,iv_component
0,a,1,4195,395,3800,0.026667,0.052667,-0.680568,0.017695
1,a,3,2848,258,2590,0.018175,0.034400,-0.637985,0.010351
2,a,2,14378,1118,13260,0.093053,0.149067,-0.471228,0.026395
3,a,4,128579,5729,122850,0.862105,0.763867,0.120984,0.011885
4,g,EC,4,2,2,0.000014,0.000267,-2.945117,0.000746
5,g,DE,9,3,6,0.000042,0.000401,-2.252016,0.000808
6,g,FR,18,3,15,0.000105,0.000401,-1.335739,0.000395
7,g,CL,9,1,8,0.000056,0.000134,-0.865732,0.000067
8,g,__OTHER__,52,5,47,0.000330,0.000668,-0.704473,0.000238
9,g,RU,73,6,67,0.000471,0.000802,-0.532250,0.000176


## Plots

This cells belwos are auxiliaries to plot some standard distributionz for the graph. 

In [12]:
col_to_plot = continuous_cols[0] # 'monto', 'score'
fig = plot_numeric_feature(df_complete_data, col_to_plot, target="fraude", nbins=40,annotations_text=df_cont_stats.loc[col_to_plot]["interpretation"])
fig.show()

In [13]:
from src.preprocessing.visualization import plot_categorical_feature

col_to_plot = categorical_cols[3] # 'b_is_missing', 'a','o_is_missing'
fig = plot_categorical_feature(df_complete_data, col_to_plot, target="fraude",annotations_text=df_cat_stats.loc[col_to_plot]["interpretation"])
fig.show()

These codes below save all the images in the respective folder 

In [14]:
# plot numerical features

save_all_feature_plots(
    df_complete_data,
    features=continuous_cols,
    dtype = "continuous",
    target="fraude",
    out_dir=IMAGES_NUMERIC_DIR,
    save_html=False,   # interactive
    save_png=True,    # static images (needs kaleido)
    annotations_text_by_feature=df_cont_stats["interpretation"].to_dict()
)

Plotting feature: b (dtype=continuous)
Plotting feature: c (dtype=continuous)
Plotting feature: d (dtype=continuous)
Plotting feature: e (dtype=continuous)
Plotting feature: f (dtype=continuous)
Plotting feature: h (dtype=continuous)
Plotting feature: l (dtype=continuous)
Plotting feature: m (dtype=continuous)
Plotting feature: monto (dtype=continuous)
Plotting feature: score (dtype=continuous)
Plotting feature: j_freq_encoded (dtype=continuous)
Plotting feature: hour_sin (dtype=continuous)
Plotting feature: hour_cos (dtype=continuous)
Plotting feature: monto_per_e (dtype=continuous)
Plotting feature: log_e (dtype=continuous)
Plotting feature: log_f (dtype=continuous)
Plotting feature: log_monto (dtype=continuous)
Plotting feature: yeojohnson_e (dtype=continuous)
Plotting feature: yeojohnson_f (dtype=continuous)
Plotting feature: boxcox_e (dtype=continuous)
Plotting feature: boxcox_f (dtype=continuous)
Plotting feature: boxcox_monto (dtype=continuous)
Plotting feature: interaction_mont

In [15]:
# plot numerical features

save_all_feature_plots(
    df_complete_data,
    features=categorical_cols,
    target="fraude",
    dtype = "categorical",
    out_dir=IMAGES_CATEGORY_DIR,
    save_html=False,   # interactive
    save_png=True,    # static images (needs kaleido)
    annotations_text_by_feature=df_cat_stats["interpretation"].to_dict()
)

Plotting feature: a (dtype=categorical)
Plotting feature: g (dtype=categorical)
Plotting feature: j (dtype=categorical)
Plotting feature: n (dtype=categorical)
Plotting feature: o (dtype=categorical)
Plotting feature: p (dtype=categorical)
Plotting feature: b_is_missing (dtype=categorical)
Plotting feature: c_is_missing (dtype=categorical)
Plotting feature: o_is_missing (dtype=categorical)
Plotting feature: e_is_zero (dtype=categorical)
Plotting feature: f_is_zero (dtype=categorical)
Plotting feature: m_is_zero (dtype=categorical)
Plotting feature: h_is_zero (dtype=categorical)
Plotting feature: c_top1_flag (dtype=categorical)
Plotting feature: e_top1_flag (dtype=categorical)
Plotting feature: f_top1_flag (dtype=categorical)
Plotting feature: monto_top1_flag (dtype=categorical)
Plotting feature: bin_c (dtype=categorical)
Plotting feature: bin_d (dtype=categorical)
Plotting feature: bin_e (dtype=categorical)
Plotting feature: bin_f (dtype=categorical)
Plotting feature: bin_h (dtype=cate

# General Results


An analysis of the continuous variables reveals several key insights for feature selection:

**Continuos Variables**

- *Transformed vs. Raw Features*: Features with transformations to reduce skew like ``e``, ``log_e``, ``boxcox_e``, and ``yeojohnson_e`` all show similar predictive performance. This indicates that while transformations are crucial for stabilizing linear models, they do not inherently create a stronger signal than the raw variable. For a linear model, it would be beneficial to select one of these transformed versions (e.g., yeojohnson_e) instead of the raw e. For a tree-based model, using the original feature e is sufficient, as trees can handle the skewness.

- *Cyclic Time Features*: The engineered features ``hour_sin`` and ``hour_cos``, often suggested in literature for temporal data, demonstrated very weak predictive power with AUCs of 0.505 and 0.501 respectively. They are not strong predictors in this dataset and will not be considered for further analysis.

- *PCA Components*: The principal components ``pca_beh_1`` (from d and m) and ``pca_amt_1`` (from e and monto) proved to be relevant. ``pca_beh_1`` had a KS of 0.166 and an AUC of 0.615, while ``pca_amt_1`` had a KS of 0.101 and an AUC of 0.561. These components successfully capture variance and could be valuable for a linear model by reducing multicollinearity.

- *Score Variable*: The score feature was highly correlated with fraud, showing a strong AUC of 0.669. This is likely a pre-calculated risk score from another model, and its high predictive power confirms its utility as a key feature.

- *General Feature Impact*: Based on their absolute AUC values, the ascending order of predictive impact for the original continuous variables is: ``c`` (0.511), ``h`` (0.519), ``b`` (0.562), ``e`` (0.562), ``m`` (0.592), ``d`` (0.615), ``l`` (0.621), and ``f`` (0.622).

- *Negative Correlation of Feature d*: The feature d has a notable negative Point-Biserial correlation (r_pb = -0.078) and a respectable absolute AUC of 0.615. This indicates that as the value of d increases, the probability of fraud tends to decrease, making it a valuable risk-mitigating indicator.

- *Feature f as the Strongest Original Predictor*: Among the original, non-score variables, feature f stands out with the highest predictive power, showing an absolute AUC of 0.622 and a KS statistic of 0.11. This makes it the most impactful raw feature for separating fraud from non-fraud.

- *Weakness of Feature c*: With an AUC of just 0.511, feature `c` is barely better than a random guess. It has the lowest predictive power of all continuous variables and is a strong candidate for removal from any model to reduce noise.

- *Limited Value of Simple Interactions*: The engineered features `monto_per_e` (ratio) and `interaction_monto_e` (multiplication) did not provide a significant predictive lift over the original e feature, both having an AUC of around 0.56. This suggests these specific simple interactions do not capture new, valuable information.

- *Conclusion*: For a tree-based model, we can keep all original features as they are robust to skewness and correlation. For a linear regression model, it would be best to use transformed versions of these features (e.g., log_f, yeojohnson_e) and the PCA components to create a more stable and predictive model. The final selection of transformations should be evaluated during the modeling phase.



**Categorical Features**

-   *Temporal Features (`is_weekend`, `weekday`):* These features were not found in the final statistics file. However, based on the very low predictive power of the continuous time features (`hour_sin`, `hour_cos`), it is likely that simple weekend or weekday flags also have little to no predictive impact on fraud.

-   *Zero-Value Indicators:* The `e_is_zero` flag is not a relevant predictor, with an Information Value (IV) of only 0.012 ("Useless"). In contrast, the flags for zero values in other features, such as `f_is_zero` (IV=0.049) and `m_is_zero` (IV=0.048), show a weak but more noticeable predictive signal.

-   *Missing Value Indicators:* The `b_is_missing` and `c_is_missing` flags were not relevant, both showing a negligible IV of 0.007. However, `o_is_missing` is one of the most powerful predictors in the entire dataset, with a very strong IV of 0.464. This indicates that the absence of a value in feature `o` is highly correlated with fraud.

-   *Binned Continuous Variables:* Even though the original continuous variable `c` was one of the worst predictors (AUC=0.511), its binned version, `bin_c`, demonstrates a moderate predictive impact with an IV of 0.103 ("Medium"). This shows that binning can successfully capture non-linear relationships that the raw variable misses. This is a key insight: for linear models, binning can unlock predictive power by capturing non-linear patterns that the raw continuous variable misses. This strategy should be considered for other continuous variables as well.

-   *Rare Grouping:* Grouping rare categories for features `a` and `j` did not significantly change their predictive power compared to their original forms. For instance, the original `a` has an IV of 0.066, and `a_rare_grouped` has a nearly identical IV. This suggests that for these variables, the rare categories themselves do not carry a distinct fraud signal.

-   *The Power of Binary Flags:* The features `n` and `p` are simple binary categorical variables, yet they are some of the strongest predictors. `n` has a "Strong" IV of 0.361 and `p` has a "Medium" IV of 0.246. This highlights that simple, pre-existing flags in the data can be more powerful than complex engineered features. They should be prioritized for inclusion in any model.

-   *`o_is_missing` is a "Super-Predictor":* With an IV of 0.464, the `o_is_missing` flag is on the cusp of being "suspiciously" high. This means the absence of data in column `o` is a very strong indicator of fraud. While extremely valuable, it's worth double-checking that this isn't a form of data leakage (i.e., a variable that would not be available at the time of prediction in a real-world scenario).

-   *Geographic and High-Cardinality Features are Weak:* The geographic feature `g` (IV=0.069) and the high-cardinality feature `j` (IV=0.082) are both classified as "Weak" predictors. While they show a statistically significant relationship to fraud, their predictive power is limited. They are unlikely to be primary drivers in a model but could provide marginal gains.


# Manual Feature Descriptions Dataset Creation

This table was built manually, based on the statistical results and visual observations from the analysis above. Each feature was reviewed individually and assigned a recommendation based on its predictive power, data type, and modeling context.

The file is saved as `phase_02_manual_features_descriptions.csv` in the results directory and can be loaded via `load_features_descriptions()` from `src.etl`.

This is the content of the table:

| Variable | Source | dtype | Recommend Use | Use in Tree Model | Use in Linear Model | Observations |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| a | Original | categorical | Yes | Yes | Yes | Weak predictor (IV=0.066). |
| b | Original | continuous | Yes | Yes | Yes | Weak predictor (AUC=0.562). |
| c | Original | continuous | No | No | No | Very weak predictor, almost random (AUC=0.511). |
| d | Original | continuous | Yes | Yes | Yes | Good predictor (AUC=0.615) with negative correlation. |
| e | Original | continuous | Yes | Yes | No (use transform) | Base for many transformations. Skewed. |
| f | Original | continuous | Yes | Yes | No (use transform) | Strongest original predictor (AUC=0.622). |
| g | Original | categorical | Yes | Yes | Yes | Weak geographic predictor (IV=0.069). |
| h | Original | continuous | No | No | No | Very weak predictor (AUC=0.519). |
| j | Original | categorical | Yes | Yes | Yes | Weak high-cardinality predictor (IV=0.082). |
| l | Original | continuous | Yes | Yes | Yes | Strong predictor (AUC=0.621). |
| m | Original | continuous | Yes | Yes | Yes | Moderate predictor (AUC=0.592). |
| n | Original | categorical | Yes | Mandatory | Mandatory | Strong binary predictor (IV=0.361). |
| o | Original | categorical | Yes | Yes | Yes | Base for the powerful `o_is_missing` feature. |
| p | Original | categorical | Yes | Mandatory | Mandatory | Medium-strength binary predictor (IV=0.246). |
| fecha | Original | datetime | Yes | Yes | Yes | Base for temporal features. |
| monto | Original | continuous | Yes | Yes | No (use transform) | Core economic variable, heavy-tailed. |
| score | Original | continuous | Yes | Mandatory | Mandatory | Strongest single predictor (AUC=0.669). Likely pre-computed. |
| b_is_missing | Engineered | binary | No | No | No | Useless predictor (IV=0.007). |
| c_is_missing | Engineered | binary | No | No | No | Useless predictor (IV=0.007). |
| o_is_missing | Engineered | binary | Yes | Mandatory | Mandatory | Very strong, "super-predictor" (IV=0.464). Potentially suspicious. |
| e_is_zero | Engineered | binary | No | No | No | Useless predictor (IV=0.012). |
| f_is_zero | Engineered | binary | Yes | Yes | Yes | Weak but useful signal (IV=0.049). |
| m_is_zero | Engineered | binary | Yes | Yes | Yes | Weak but useful signal (IV=0.048). |
| h_is_zero | Engineered | binary | No | No | No | Useless predictor (IV=0.002). |
| c_top1_flag | Engineered | binary | No | No | No | Useless predictor (IV=0.007). |
| e_top1_flag | Engineered | binary | No | No | No | Useless predictor (IV=0.001). |
| f_top1_flag | Engineered | binary | No | No | No | Useless predictor (IV=0.011). |
| monto_top1_flag | Engineered | binary | Yes | Yes | Yes | Weak predictor (IV=0.023). |
| bin_c | Engineered | categorical | Yes | Not Necessary | Yes | Medium predictor (IV=0.103). Unlocks value from weak raw `c`. |
| bin_d | Engineered | categorical | Yes | Not Necessary | Yes | Strong predictor (IV=0.309). |
| bin_e | Engineered | categorical | Yes | Not Necessary | Yes | Weak predictor (IV=0.081). |
| bin_f | Engineered | categorical | Yes | Not Necessary | Yes | Strong predictor (IV=0.320). |
| bin_h | Engineered | categorical | No | No | No | Useless predictor (IV=0.007). |
| bin_k | Engineered | categorical | No | No | No | Based on identifier `k`. Data leakage risk. |
| bin_l | Engineered | categorical | Yes | Not Necessary | Yes | Strong predictor (IV=0.324). |
| bin_m | Engineered | categorical | Yes | Not Necessary | Yes | Medium predictor (IV=0.218). |
| bin_score | Engineered | categorical | Yes | Not Necessary | Yes | Strong predictor (IV=0.439). |
| bin_monto | Engineered | categorical | Yes | Not Necessary | Yes | Medium predictor (IV=0.101). |
| a_rare_grouped | Engineered | categorical | Yes | Yes | Yes | No significant improvement over original `a`. |
| j_rare_grouped | Engineered | categorical | Yes | Yes | Yes | No significant improvement over original `j`. |
| j_freq_encoded | Engineered | continuous | Yes | Yes | Yes | Alternative to `j_rare_grouped`. Weak predictor (AUC=0.542). |
| hour | Engineered | categorical | No | No | No | Weak predictor (IV=0.013). |
| weekday | Engineered | categorical | No | No | No | Useless predictor (IV=0.001). |
| is_weekend | Engineered | binary | No | No | No | Useless predictor (IV=0.000). |
| hour_sin | Engineered | continuous | No | No | No | Useless predictor (AUC=0.505). |
| hour_cos | Engineered | continuous | No | No | No | Useless predictor (AUC=0.501). |
| monto_per_e | Engineered | continuous | No | No | No | No improvement over original `e` (AUC=0.561). |
| log_e | Engineered | continuous | Yes | Not Necessary | Yes | Good transformation for `e`. |
| log_f | Engineered | continuous | Yes | Not Necessary | Yes | Good transformation for `f`. |
| log_monto | Engineered | continuous | Yes | Not Necessary | Yes | Good transformation for `monto`. |
| yeojohnson_e | Engineered | continuous | Yes | Not Necessary | Yes | Good transformation for `e`. |
| yeojohnson_f | Engineered | continuous | Yes | Not Necessary | Yes | Good transformation for `f`. |
| boxcox_e | Engineered | continuous | Yes | Not Necessary | Yes | Good transformation for `e`. |
| boxcox_f | Engineered | continuous | Yes | Not Necessary | Yes | Good transformation for `f`. |
| boxcox_monto | Engineered | continuous | Yes | Not Necessary | Yes | Good transformation for `monto`. |
| interaction_monto_e | Engineered | continuous | No | No | No | No improvement over original `e` (AUC=0.562). |
| pca_beh_1 | Engineered | continuous | Yes | Yes | Yes | Good predictor (AUC=0.615). Reduces multicollinearity. |
| pca_beh_2 | Engineered | continuous | No | No | No | Useless predictor (AUC=0.501). |
| pca_amt_1 | Engineered | continuous | Yes | Yes | Yes | Weak predictor (AUC=0.561). Reduces multicollinearity. |
| pca_amt_2 | Engineered | continuous | No | No | No | Useless predictor (AUC=0.500). |

# Next steps:

Based on the analysis of features, and the previous results, we can still continue with the initial plan for next phase:

### Phase 3: Model Selection & Profit Optimization
* **Algorithm Candidates:** Train high-performance classifiers (e.g., Random Forest, XGBoost) and establish a baseline using Logistic Regression.
* **Cost-Sensitive Learning:** Incorporate the 4:1 loss ratio ($100\%$ loss vs. $25\%$ margin) directly into the model training weights or cost function.
* **Threshold Optimization:** Instead of using the default 0.5 threshold, identify the specific probability cutoff (theoretically **0.20**) that maximizes the Profit Function.

**Iterative Refinement:** At the conclusion of each phase, the strategy will be updated based on data findings. The final output will be a production-ready ML model optimized for **Maximum Net Profit**.